# 03/ test SNCadenceMetric

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-05
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/science/Number_SNeIa_metric.ipynb

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
import rubin_sim.maf as maf
# import rubin_sim.utils as rsUtils

try:
    from rubin_sim.data import get_baseline
except ImportError:
    from rubin_scheduler.data import get_baseline

import healpy as hp

## Configuration

In [ ]:
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
baseline_file = get_baseline()
runname = baseline_file.split("/")[-1].replace(".db", "")
print(runname)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="03_maf_testSNCadence_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## SNCadenceMetric

In [ ]:
# to view the signature of the Metrics class
%pinfo maf.SNCadenceMetric

In [ ]:
# to view the code of the metrics
%psource maf.SNCadenceMetric

### Season n'existe pas ==> Stacker

In [ ]:
# from rubin_sim.maf.stackers import SeasonStacker
# stacker = SeasonStacker()

from rubin_sim.maf.stackers import BaseStacker


class MySeasonStacker(BaseStacker):
    cols_added = ["season"]
    cols_req = ["observationStartMJD"]  # 🔴 OBLIGATOIRE

    def __init__(self, mjdCol="observationStartMJD"):
        self.mjdCol = mjdCol

    def _run(self, simData, cols_present=False):
        mjd0 = simData[self.mjdCol].min()
        season = np.floor((simData[self.mjdCol] - mjd0) / 365.25)

        simData["season"] = season.astype(int)
        return simData

In [ ]:
import rubin_sim.maf as maf


class SNCadenceMetricPatched(maf.metrics.SNCadenceMetric):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        # 🔴 enlever season partout
        if hasattr(self, "cols_req"):
            self.cols_req = [c for c in self.cols_req if c != "season"]

        if hasattr(self, "col_name_arr"):
            self.col_name_arr = [c for c in self.col_name_arr if c != "season"]

In [ ]:
bundle_list = []

sne_nside = 16
sqlconstraint = ""
stacker = MySeasonStacker()
sn_summary = [maf.metrics.MedianMetric(), maf.metrics.SumMetric(), maf.metrics.MeanMetric()]
slicer = maf.slicers.HealpixSlicer(nside=sne_nside, use_cache=False)
metric = SNCadenceMetricPatched()
bundle = maf.metric_bundle.MetricBundle(
    metric, slicer, sqlconstraint, stacker_list=[stacker], summary_metrics=sn_summary
)

### Create group bundle

In [ ]:
bdict = {"sne": bundle}
group = maf.MetricBundleGroup(bdict, baseline_file, out_dir, resultsDb)

In [ ]:
group.run_all()
group.plot_all(closefigs=False)

In [ ]:
fig = plt.figure(figsize=(8, 6))
bundle.plot(plotFunc=maf.plots.HealpixSkyMap())
plt.title("SNCadenceMetric (zlim)")

In [ ]:
fig = plt.figure(figsize=(6, 4))
bundle.plot(plotFunc=maf.plots.HealpixHistogram())
plt.title("Distribution SNCadenceMetric")

In [ ]:
plotDict = {"colorMin": 0.0, "colorMax": 0.8}

fig = plt.figure(figsize=(8, 6))
bundle.plot(plotFunc=maf.plots.HealpixSkyMap(), plotDict=plotDict)
plt.title("SNCadenceMetric (fixed scale)")

In [ ]:
print("Mean:", bundle.metricValues.mean())
print("Median:", np.median(bundle.metricValues))
print("Min:", bundle.metricValues.min())
print("Max:", bundle.metricValues.max())